

# **Laboratorio 11: Pienso, luego predigo 💡**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### **Equipo: poplolitas - notebooks sin nombre no serán revisados**

- Nombre de alumno 1: Javiera Arévalo D.
- Nombre de alumno 2: Laura Maldonado L.

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/lauraflm/MDS7202-Laboratorio-de-Programacion-Cientifica-para-Ciencia-de-Datos)

## **Temas a tratar**

- Reinforcement Learning
- Large Language Models

## **Reglas:**

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Objetivos principales del laboratorio**

- Resolución de problemas secuenciales usando Reinforcement Learning
- Habilitar un Chatbot para entregar respuestas útiles usando Large Language Models.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

## **1. Reinforcement Learning (2.0 puntos)**

En esta sección van a usar métodos de RL para resolver dos problemas interesantes: `Blackjack` y `LunarLander`.

In [ ]:
!pip install -qqq gymnasium stable_baselines3
!pip install -qqq swig
!pip install -qqq gymnasium[box2d]
!pip install gymnasium

In [ ]:
!pip install -qqq gymnasium stable_baselines3 swig gymnasium[box2d]
import gymnasium as gym
import numpy as np
from gymnasium.spaces import MultiDiscrete
from stable_baselines3 import A2C, PPO, DQN


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### **1.1 Blackjack (1.0 puntos)**

<p align="center">
  <img src="https://www.recreoviral.com/wp-content/uploads/2016/08/s3.amazonaws.com-Math.gif"
" width="400">
</p>

La idea de esta subsección es que puedan implementar métodos de RL y así generar una estrategia para jugar el clásico juego Blackjack y de paso puedan ~~hacerse millonarios~~ aprender a resolver problemas mediante RL.

Comencemos primero preparando el ambiente. El siguiente bloque de código transforma las observaciones del ambiente a `np.array`:


In [ ]:
from gymnasium.spaces import Box

class FlattenObservation(gym.ObservationWrapper):
    def __init__(self, env):
        super(FlattenObservation, self).__init__(env)
        self.observation_space = MultiDiscrete(np.array([32, 11, 2]))

    def observation(self, observation):
        return np.array(observation).flatten()

# Create and wrap the environment
env = gym.make("Blackjack-v1")
env = FlattenObservation(env)

#### **1.1.1 Descripción de MDP (0.2 puntos)**

Entregue una breve descripción sobre el ambiente [Blackjack](https://gymnasium.farama.org/environments/toy_text/blackjack/) y su formulación en MDP, distinguiendo de forma clara y concisa los estados, acciones y recompensas.

`Respuesta`


El ambiente Blackjack es una simulación de un juego de cartas que tiene como objetivo aprender RL. Siguiendo con lo anterior, el foco del agente es aprender a jugar Blackjack tomando decisiones de Hit (pedir carta) o Stick (quedarse).

La composición del MDP es estados, acciones, recompensas y dinámicas. Estas se detallarán en los siguientes puntos:

*   Estado: tiene tres elementos (player_sum, dealer_showing, usable_ace)

*   Acciones: Hit (pedir carta) o Stick (quedarse)

*   Recompensas: Ganas (+1), pierdes (-1) o empatas (0)

*   Dinámicas: se tiene el inicio del juego, el comportamiento de agente, el comportamiento del dealer y el final del episodio.



Entonces, el agente debe aprender a maximizar el valor esperado de la recompensa, esto se separa en,
distinguir cúando conviene pedir una carta, cuándo conviene quedarse, cómo influye la carta del dealer y cómo usar el As (como 1 o 11).

#### **1.1.2 Generando un Baseline (0.2 puntos)**

Simule un escenario en donde se escojan acciones aleatorias. Repita esta simulación 5000 veces y reporte el promedio y desviación de las recompensas. ¿Cómo calificaría el performance de esta política? ¿Cómo podría interpretar las recompensas obtenidas?

In [ ]:
def random_policy(env, n_episodes=5000):
    todas_recompensas = []

    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0

        while not done:
            action = env.action_space.sample()
            obs, reward, terminated, truncated, _ = env.step(action)

            done = terminated or truncated
            total_reward += reward

        todas_recompensas.append(total_reward)

    return np.mean(todas_recompensas), np.std(todas_recompensas)

mean_rand, std_rand = random_policy(env)
print("Baseline (Random Policy)")
print("Promedio:", mean_rand)
print("Desviación:", std_rand)


Baseline (Random Policy)
Promedio: -0.3794
Desviación: 0.9025827607482872


¿Cómo calificaría el performance de esta política?

*   El rendimiento de esta política es bajo, esto se debe a que la recompensa promedio es negativa (–0.3794). Lo anterior indica que en promedio, el agente pierde más rondas de las que gana. Como la política elige acciones sin ningún criterio estratégico, es esperable que tenga un desempeño bajo en un juego donde decisiones afectan en la probabilidad de ganar.

¿Cómo podría interpretar las recompensas obtenidas?

*   Las recompensas indican cuanta proporción de victorias, derrotas y empates tiene el agente. Entonces, que haya una recompensa promedio negativa como se dijo anteriormente, el agente pierde más de lo que gana. Además, se tiene una desviación estándar de 0.9025 que indica que existe mucha variación entre los distintos resultados que se obtienen.
Entonces, con el valor promedio confirma que una política aleatoria no es suficiente para obtener un desempeño competitivo.

#### **1.1.3 Entrenamiento de modelo (0.2 puntos)**

A partir del siguiente [enlace](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html), escoja un modelo de `stable_baselines3` y entrenelo para resolver el ambiente `Blackjack`.

In [ ]:
model = DQN("MlpPolicy", env, verbose=0)
model.learn(total_timesteps=20_000)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


#### **1.1.4 Evaluación de modelo (0.2 puntos)**

Repita el ejercicio 1.1.2 pero utilizando el modelo entrenado. ¿Cómo es el performance de su agente? ¿Es mejor o peor que el escenario baseline?

In [ ]:
def evaluate(model, env, n_episodes=5000):
    rewards = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0
        while not done:
            action, _ = model.predict(obs)
            obs, reward, done, _, _ = env.step(action)
            total_reward += reward
        rewards.append(total_reward)
    return np.mean(rewards), np.std(rewards)

mean_model, std_model = evaluate(model, env)
print("Modelo RL")
print("Promedio:", mean_model)
print("Desv:", std_model)


Modelo RL
Promedio: -0.0986
Desv: 0.9499884420349545


¿Cómo es el performance de su agente?

* Los resultados obtenidos indican que la recompensa promedio es –0.0986 esto indica que aunque sigue perdiendo más que ganando, lo hace en menos medida. Por otro lado, la desviación estándar es de 0.9499, lo que significa que sigue existiendo mucha variabilidad entre los resultados.


¿Es mejor o peor que el escenario baseline?
* El desempeño del agente entrenado es mucho mejor que el baseline aleatorio, esto debido a que la recompensa promedio es más cercana a cero. Entonces, el modelo RL reduce la pérdida promedio y demuestra haber aprendido patrones útiles, por ejemplo cuándo evitar pedir carta en estados que sean más riesgosos.

#### **1.1.5 Estudio de acciones (0.2 puntos)**

Genere una función que reciba un estado y retorne la accion del agente. Luego, use esta función para entregar la acción escogida frente a los siguientes escenarios:

- Suma de cartas del agente es 6, dealer muestra un 7, agente no tiene tiene un as
- Suma de cartas del agente es 19, dealer muestra un 3, agente tiene tiene un as

¿Son coherentes sus acciones con las reglas del juego?

Hint: ¿A que clase de python pertenecen los estados? Pruebe a usar el método `.reset` para saberlo.

In [ ]:
def predict_action(model, state):
    state = np.array(state).flatten()
    action, _ = model.predict(state)
    return action

In [ ]:
# Caso 1
estado1 = (6, 7, 0)
print("Acción caso 1:", predict_action(model, estado1))

# Caso 2
estado2 = (19, 3, 1)
print("Acción caso 2:", predict_action(model, estado2))


Acción caso 1: 1
Acción caso 2: 0


In [ ]:
obs, _ = env.reset()
print(type(obs), obs)


<class 'numpy.ndarray'> [6 1 0]


¿Son coherentes sus acciones con las reglas del juego?

1. Caso 1:
* Estado: (suma = 6, dealer = 7, usable_ace = 0)
* Acción del agente: 1 (Hit)
Esta acción es coherente con las reglas y con la estrategia básica de Blackjack.
Con una suma baja como 6, el jugador siempre debería pedir carta, independientemente de la carta del dealer. El agente se comporta correctamente al elegir hit.

2. Caso 2:
* Estado: (suma = 19, dealer = 3, usable_ace = 1)
* Acción del agente: 0 (Stick)
Esta acción también es coherente. Con un 19 y si se incluye un As, la decisión óptima es quedarse en la gran mayoría de situaciones.
Además, la carta del dealer es un 3, lo que lo deja relativamente débil.


¿A que clase de python pertenecen los estados?
El resultado un (array de numpy) que salió indica que:
“El jugador tiene 6 puntos, el dealer muestra un 1, y el jugador no tiene un As que cuente como 11.”

### **1.2 LunarLander**

<p align="center">
  <img src="https://i.redd.it/097t6tk29zf51.jpg"
" width="400">
</p>

Similar a la sección 2.1, en esta sección usted se encargará de implementar una gente de RL que pueda resolver el ambiente `LunarLander`.

Comencemos preparando el ambiente:


In [ ]:
import gymnasium as gym
env = gym.make("LunarLander-v3", render_mode = "rgb_array", continuous = True) # notar el parámetro continuous = True

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

Noten que se especifica el parámetro `continuous = True`. ¿Que implicancias tiene esto sobre el ambiente?




Además, se le facilita la función `export_gif` para el ejercicio 2.2.4:

In [ ]:
import imageio
import numpy as np

def export_gif(model, n = 5):
  '''
  función que exporta a gif el comportamiento del agente en n episodios
  '''
  images = []
  for episode in range(n):
    obs = model.env.reset()
    img = model.env.render()
    done = False
    while not done:
      images.append(img)
      action, _ = model.predict(obs)
      obs, reward, done, info = model.env.step(action)
      img = model.env.render(mode="rgb_array")

  imageio.mimsave("agent_performance.gif", [np.array(img) for i, img in enumerate(images) if i%2 == 0], fps=29)

#### **1.2.1 Descripción de MDP (0.2 puntos)**

Entregue una breve descripción sobre el ambiente [LunarLander](https://gymnasium.farama.org/environments/box2d/lunar_lander/) y su formulación en MDP, distinguiendo de forma clara y concisa los estados, acciones y recompensas. ¿Como se distinguen las acciones de este ambiente en comparación a `Blackjack`?

Nota: recuerde que se especificó el parámetro `continuous = True`

`Respuesta`

El ambiente LunarLander es una simulación donde el agente debe aprender a controlar un módulo lunar para aterrizarlo de forma estable utilizando RL. El objetivo es regular correctamente los motores para evitar choques y lograr un aterrizaje seguro.

La formulación como MDP se compone de:

* Estado: un vector continuo de 8 elementos que describe posición, velocidad, ángulo, velocidad angular y si las piernas tocan el suelo.

* Acciones: con continuous=True, el agente controla la potencia del motor principal y de los motores laterales mediante un vector continuo. Esto contrasta con Blackjack, donde las acciones son discretas y mucho más simples.

* Recompensas: se otorgan por acercarse al centro, reducir la velocidad, mantener estabilidad y apoyar las piernas; y se penalizan el uso excesivo de motores, la inclinación y los choques. Un aterrizaje seguro da +100 y un choque −100.

* Dinámicas: el episodio inicia con el lander cayendo desde arriba, el agente ajusta motores para estabilizarse y termina cuando aterriza, se estrella o sale del área válida.

En resumen, el agente debe aprender a controlar la física del vehículo para maximizar la recompensa y lograr un aterrizaje seguro.

#### **1.2.2 Generando un Baseline (0.2 puntos)**

Simule un escenario en donde se escojan acciones aleatorias. Repita esta simulación 10 veces y reporte el promedio y desviación de las recompensas. ¿Cómo calificaría el performance de esta política?

In [ ]:
def random_baseline_lander(env, n_episodes=10):
    recompensas_totales = []

    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0.0

        while not done:
            # Acción aleatoria
            action = env.action_space.sample()

            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            total_reward += reward

        recompensas_totales.append(total_reward)

    return np.mean(recompensas_totales), np.std(recompensas_totales)

# mismo env que antes ->>
# env = gym.make("LunarLander-v2", render_mode="rgb_array", continuous=True)

mean_rand, std_rand = random_baseline_lander(env)
print("Baseline LunarLander (Random Policy)")
print("Promedio:", mean_rand)
print("Desviación:", std_rand)


Baseline LunarLander (Random Policy)
Promedio: -203.48680474946707
Desviación: 124.78404533963817


¿Cómo calificaría el performance de esta política?

El rendimiento de esta política tiene como resultado un promedio de recompensa (–203.4868) que muestra que el agente prácticamente siempre se estrella. Lo anterior, ya que utiliza los motores de manera ineficiente y se aleja del objetivo del aterrizaje seguro. Esto era esperable, ya que una política que selecciona acciones al azar no tiene ideas de estabilidad, orientación o velocidad necesarias para que no ocurra eso.

La desviación estándar alta de 124.7840 indica que existe bastante variabilidad entre resultados, algunos terminan en choques rápidos, otros duran un poco más, pero en general el desempeño es negativo y altamente inconsistente.

Por lo tanto, la política aleatoria no es capaz de resolver el ambiente y sirve únicamente como baseline comparativo.

#### **1.2.3 Entrenamiento de modelo (0.2 puntos)**

A partir del siguiente [enlace](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html), escoja un modelo de `stable_baselines3` y entrenelo para resolver el ambiente `LunarLander` **usando 10000 timesteps de entrenamiento**.

In [ ]:
from stable_baselines3 import PPO

# Creamos el modelo usando PPO y una política MLP
model_lander = PPO(
    "MlpPolicy",
    env,
    verbose=1
)

# Entrenamos por 10.000 timesteps
model_lander.learn(total_timesteps=10_000)

print("Entrenamiento completado.")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 120      |
|    ep_rew_mean     | -247     |
| time/              |          |
|    fps             | 844      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 117          |
|    ep_rew_mean          | -225         |
| time/                   |              |
|    fps                  | 599          |
|    iterations           | 2            |
|    time_elapsed         | 6            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0062358426 |
|    clip_fraction        | 0.0478       |
|    clip_range           | 0.2          |
|    en

#### **1.2.4 Evaluación de modelo (0.2 puntos)**

Repita el ejercicio 1.2.2 pero utilizando el modelo entrenado. ¿Cómo es el performance de su agente? ¿Es mejor o peor que el escenario baseline?

In [ ]:
def evaluate_lander_model(model, env, n_episodes=10):
    recompensas_totales = []

    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0.0

        while not done:
            # Acción del modelo (no aleatoria)
            action, _ = model.predict(obs, deterministic=True)

            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            total_reward += reward

        recompensas_totales.append(total_reward)

    return np.mean(recompensas_totales), np.std(recompensas_totales)

mean_model, std_model = evaluate_lander_model(model_lander, env)
print("Modelo LunarLander")
print("Promedio:", mean_model)
print("Desviación:", std_model)


Modelo LunarLander
Promedio: -126.81622567346315
Desviación: 85.8330382583272


¿Cómo es el performance de su agente?

* El desempeño del agente entrenado es mejor que el de la política aleatoria. Su promedio pasó de aproximadamente –203 (baseline) a –126, lo que muestra que el modelo aprendió a controlar, ya que reduce choques rápidos, tiene más estabilidad y utiliza los motores de manera más eficiente. Aunque el rendimiento sigue siendo negativo, el agente ya no actúa de forma completamente errática y empieza a realizar maniobras más razonables.

La desviación estándar también disminuyó (de 124.78 a 85.83), lo que indica que el comportamiento del agente es un poco más consistente, aunque todavía variable debido a la complejidad del entorno.

¿Es mejor o peor que el escenario baseline?

* En base a lo anterior, se puede concluir que aunque este escenario no es perfecto, si tiene mejores resultados en comparación al baseline. Se concluye que el entrenamiento ya mejora la supervivencia y el control del agente respecto al juego completamente aleatorio.

#### **1.2.5 Optimización de modelo (0.2 puntos)**

Repita los ejercicios 1.2.3 y 1.2.4 hasta obtener un nivel de recompensas promedio mayor a 50. Para esto, puede cambiar manualmente parámetros como:
- `total_timesteps`
- `learning_rate`
- `batch_size`

Una vez optimizado el modelo, use la función `export_gif` para estudiar el comportamiento de su agente en la resolución del ambiente y comente sobre sus resultados.

Adjunte el gif generado en su entrega (mejor aún si además adjuntan el gif en el markdown).

In [ ]:
from stable_baselines3 import PPO

# Modelo "optimizado" con otros hiperparámetros
model_opt = PPO(
    "MlpPolicy",
    env,
    learning_rate=1e-4,   # más pequeño → aprendizaje más estable
    batch_size=128,       # batch un poco mayor
    n_steps=2048,         # tamaño de los rollouts
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1
)

# Entrenamos con más timesteps
model_opt.learn(total_timesteps=200_000)

print("Entrenamiento optimizado completado.")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 97.1     |
|    ep_rew_mean     | -243     |
| time/              |          |
|    fps             | 928      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 107          |
|    ep_rew_mean          | -265         |
| time/                   |              |
|    fps                  | 714          |
|    iterations           | 2            |
|    time_elapsed         | 5            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0008480201 |
|    clip_fraction        | 4.88e-05     |
|    clip_range           | 0.2          |
|    en

In [ ]:
mean_opt, std_opt = evaluate_lander_model(model_opt, env)
print("Modelo optimizado LunarLander")
print("Promedio:", mean_opt)
print("Desviación:", std_opt)


Modelo optimizado LunarLander
Promedio: -22.82538807530436
Desviación: 17.136774185507367


Tras optimizar los hiperparámetros, el desempeño del agente mejoró significativamente: pasó de recompensas muy negativas a un promedio cercano a –22.82, lo que indica que ahora controla mejor el módulo y mantiene una trayectoria más estable.

Al observar el GIF generado, se ve que el lander desciende de forma más suave, corrige mejor su orientación y se acerca con más control a la zona de aterrizaje. Aunque todavía no logra un aterrizaje perfectamente seguro en todos los episodios, el comportamiento es claramente más estable que en las versiones anteriores.

In [ ]:
import gymnasium as gym
import imageio
import numpy as np

def export_gif(model, n=5):
    # Crear un nuevo entorno SOLO para grabar el gif
    env = gym.make("LunarLander-v3", render_mode="rgb_array", continuous=True)

    images = []
    for episode in range(n):
        obs, _ = env.reset()
        done = False

        while not done:
            # Render actual (frame)
            img = env.render()
            images.append(img)

            # Acción del modelo
            action, _ = model.predict(obs, deterministic=True)

            # Paso del entorno (API gymnasium)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

    env.close()

    imageio.mimsave(
        "agent_performance.gif",
        [np.array(img) for i, img in enumerate(images) if i % 2 == 0],
        fps=29
    )

# Llamada:
export_gif(model_opt, n=5)
print("GIF generado: agent_performance.gif")


/usr/local/lib/python3.12/dist-packages/imageio/plugins/pillow.py:410: DeprecationWarning: The keyword `fps` is no longer supported. Use `duration`(in ms) instead, e.g. `fps=50` == `duration=20` (1000 * 1/50).
  warnings.warn(


GIF generado: agent_performance.gif


<img src="/content/agent_performance.gif" width="350">


## **2. Large Language Models (4.0 puntos)**

En esta sección se enfocarán en habilitar un Chatbot que nos permita responder preguntas útiles a través de LLMs.

### **2.0 Configuración Inicial**

<p align="center">
  <img src="https://media1.tenor.com/m/uqAs9atZH58AAAAd/config-config-issue.gif"
" width="400">
</p>

Como siempre, cargamos todas nuestras API KEY al entorno:

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

Enter your Google AI API key: ··········
Enter your Tavily API key: ··········


### **2.1 Retrieval Augmented Generation (1.5 puntos)**

<p align="center">
  <img src="https://y.yarn.co/218aaa02-c47e-4ec9-b1c9-07792a06a88f_text.gif"
" width="400">
</p>

El objetivo de esta subsección es que habiliten un chatbot que pueda responder preguntas usando información contenida en documentos PDF a través de **Retrieval Augmented Generation.**

#### **2.1.1 Reunir Documentos (0 puntos)**

Reuna documentos PDF sobre los que hacer preguntas siguiendo las siguientes instrucciones:
  - 2 documentos .pdf como mínimo.
  - 50 páginas de contenido como mínimo entre todos los documentos.
  - Ideas para documentos: Documentos relacionados a temas académicos, laborales o de ocio. Aprovechen este ejercicio para construir algo útil y/o relevante para ustedes!
  - Deben ocupar documentos reales, no pueden utilizar los mismos de la clase.
  - Deben registrar sus documentos en la siguiente [planilla](https://docs.google.com/spreadsheets/d/1Hy1w_dOiG2UCHJ8muyxhdKPZEPrrL7BNHm6E90imIIM/edit?usp=sharing). **NO PUEDEN USAR LOS MISMOS DOCUMENTOS QUE OTRO GRUPO**
  - **Recuerden adjuntar los documentos en su entrega**.

In [ ]:
%pip install --upgrade --quiet PyPDF2

In [ ]:
import PyPDF2

doc_paths = [
    "/content/Dimova-Emili.pdf",
    "/content/Exploring the Romantic Comedy_ From the 90s to Today.pdf"
]

assert len(doc_paths) >= 2, "Deben adjuntar un mínimo de 2 documentos"
total_paginas = sum(len(PyPDF2.PdfReader(open(doc, "rb")).pages) for doc in doc_paths)
assert total_paginas >= 50, f"Páginas insuficientes: {total_paginas}"


#### **2.1.2 Vectorizar Documentos (0.2 puntos)**

Vectorice los documentos y almacene sus representaciones de manera acorde.

In [ ]:
%pip install -q pypdf sentence-transformers faiss-cpu

In [ ]:
import pypdf

def extraer_texto_de_pdfs(doc_paths):
    textos = []
    for path in doc_paths:
        reader = pypdf.PdfReader(path)
        for page in reader.pages:
            text = page.extract_text()
            if text:
                textos.append(text)
    return "\n".join(textos)

texto_completo = extraer_texto_de_pdfs(doc_paths)
print("Longitud total del texto:", len(texto_completo))


Longitud total del texto: 290969


Longitud total del texto: 290969

In [ ]:
def crear_chunks(texto, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(texto):
        end = start + chunk_size
        chunk = texto[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = crear_chunks(texto_completo, chunk_size=1000, overlap=200)
print("Número de chunks generados:", len(chunks))


Número de chunks generados: 364


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Modelo de embeddings
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Embeddings de todos los chunks
emb_matrix = embed_model.encode(
    chunks,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Shape de la matriz de embeddings:", emb_matrix.shape)

# Crear índice FAISS
dimension = emb_matrix.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(emb_matrix)

print("Número de vectores en el índice:", index.ntotal)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Shape de la matriz de embeddings: (364, 384)
Número de vectores en el índice: 364


In [ ]:
import pickle

# Guardar índice FAISS
faiss.write_index(index, "rag_index.faiss")

# Guardar los textos de los chunks
with open("rag_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Embeddings e índice guardados en 'rag_index.faiss' y 'rag_chunks.pkl'.")


Embeddings e índice guardados en 'rag_index.faiss' y 'rag_chunks.pkl'.


#### **2.1.3 Habilitar RAG (0.3 puntos)**

Habilite la solución RAG a través de una *chain* y guárdela en una variable.

In [ ]:
import os
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

llm = genai.GenerativeModel("models/gemini-pro-latest")

In [ ]:
template = """
Eres un asistente útil que responde en español usando solo la información del contexto.

Contexto:
{context}

Pregunta:
{question}

Respuesta en español, clara y concisa:
"""

In [ ]:
def rag_chain(question: str, k: int = 4) -> str:
    # Embedding de la pregunta
    q_emb = embed_model.encode([question], convert_to_numpy=True)

    # Búsqueda en FAISS de los k chunks más cercanos
    distances, indices = index.search(q_emb, k)
    retrieved_chunks = [chunks[i] for i in indices[0]]

    # Construir el contexto
    context = "\n\n".join(retrieved_chunks)

    # Construir el prompt final
    prompt = template.format(context=context, question=question)

    # Llamar a Gemini
    response = llm.generate_content(prompt)

    # Devolver solo el texto
    return response.text

In [ ]:
pregunta = "¿Qué se comenta sobre Comedy en estos documentos?"
respuesta = rag_chain(pregunta)
print(respuesta)

En el contexto se menciona una tesis de honor de 2020 escrita por Abigail Sherlock titulada "Exploring the Romantic Comedy: From the 90s to Today".


#### **2.1.4 Verificación de respuestas (0.5 puntos)**

Genere un listado de 3 tuplas ("pregunta", "respuesta correcta") y analice la respuesta de su solución para cada una. ¿Su solución RAG entrega las respuestas que esperaba?

Ejemplo de tupla:
- Pregunta: ¿Quién es el presidente de Chile?
- Respuesta correcta: El presidente de Chile es Gabriel Boric

In [ ]:
import pypdf

reader = pypdf.PdfReader("/content/Dimova-Emili.pdf")

for i in range(3):  # leer primeras 3 páginas
    print(f"--- Página {i+1} ---")
    print(reader.pages[i].extract_text())
    print("\n\n")


--- Página 1 ---
 
 
 
Breaking taboos with “Sex and the City” 
A quantitative research of media effects of a groundbreaking TV show on perceptions of 
women 
 
 
  
 
 
Student Name: Emili Dimova 
Student Number: 379982 
 
Supervisor:   Dr. J. Lee 
 
 
Master in Media & Business 
Erasmus School of History, Culture and Communication 
Erasmus University Rotterdam 
 
 
Master Thesis  
June 22nd  
  



--- Página 2 ---
  
 
2 
 
 
ABSTRACT  
Television is still a significant force of popular culture, however continues to represent even the 
“modern” woman in a rather superficial and biased manner, both regarding her professional 
development and her intimate life. Failing to capture the countless shifts towards a more 
empowered view on women’s work achievements and expression of sexuality, such 
conservative female portrayal continues to reinforce dated understandings on the matter. The 
TV show “Sex and the City” has been recognized world-widely for its daring representation of 
the mo

In [ ]:
reader = pypdf.PdfReader("/content/Exploring the Romantic Comedy_ From the 90s to Today.pdf")

for i in range(2):
    print(f"--- Página {i+1} ---")
    print(reader.pages[i].extract_text())
    print("\n\n")


--- Página 1 ---
Honors Thesis Honors Program 
5-4-2020 
Exploring the Romantic Comedy: From the 90s to Today Exploring the Romantic Comedy: From the 90s to Today 
Abigail Sherlock 
asherlock99@gmail.com 
Follow this and additional works at: https://digitalcommons.lmu.edu/honors-thesis 
 Part of the Other Film and Media Studies Commons 
Recommended Citation Recommended Citation 
Sherlock, Abigail, "Exploring the Romantic Comedy: From the 90s to Today" (2020). Honors Thesis. 417. 
https://digitalcommons.lmu.edu/honors-thesis/417 
This Honors Thesis is brought to you for free and open access by the Honors Program at Digital Commons @ 
Loyola Marymount University and Loyola Law School. It has been accepted for inclusion in Honors Thesis by an 
authorized administrator of Digital Commons@Loyola Marymount University and Loyola Law School. For more 
information, please contact digitalcommons@lmu.edu. 



--- Página 2 ---

ɪ
ɪ
&YQMPSJOHUIF3PNBOUJD$PNFEZ'SPNUIFɫ
TUPUPEBZɫ
ɪ
ɪ
"UIFT

In [ ]:
evaluacion_rag = [
    ("¿Qué representa la serie 'Sex and the City' según el texto?",
        "El texto la describe como una representación innovadora de la mujer moderna, caracterizada por independencia financiera, libertad sexual y un rol más empoderado que desafía los estereotipos tradicionales."),
    ("¿Qué efectos menciona el documento sobre la influencia de 'Sex and the City' en las mujeres?",
        "El documento indica que la exposición a la serie incrementó las actitudes igualitarias y la percepción de empoderamiento personal entre las espectadoras."),
    ("¿Qué enfoque tiene el documento sobre las comedias románticas desde los años 90 hasta hoy?",
        "El texto analiza cómo las comedias románticas han evolucionado desde fórmulas tradicionales hacia historias más diversas, con personajes femeninos más complejos y temáticas adaptadas a cambios sociales contemporáneos.")]


In [ ]:
for pregunta, respuesta_correcta in evaluacion_rag:
    print("Pregunta:", pregunta)
    print("Respuesta esperada:", respuesta_correcta)
    print("Respuesta RAG:", rag_chain(pregunta))
    print("\n" + "-"*80 + "\n")


Pregunta: ¿Qué representa la serie 'Sex and the City' según el texto?
Respuesta esperada: El texto la describe como una representación innovadora de la mujer moderna, caracterizada por independencia financiera, libertad sexual y un rol más empoderado que desafía los estereotipos tradicionales.
Respuesta RAG: Según el texto, la serie 'Sex and the City' es una representación de la mujer moderna, no tradicional, financieramente independiente y sexualmente empoderada. También se la define como una representación de la segunda ola del feminismo que desafió las visiones más tradicionales sobre las mujeres en la cultura popular.

--------------------------------------------------------------------------------

Pregunta: ¿Qué efectos menciona el documento sobre la influencia de 'Sex and the City' en las mujeres?
Respuesta esperada: El documento indica que la exposición a la serie incrementó las actitudes igualitarias y la percepción de empoderamiento personal entre las espectadoras.
Respuesta 

Para evaluar el desempeño del RAG, se definieron tres preguntas basadas en conceptos que se encuentran presentes en los PDFs utilizados (Sex and the City y la evolución de las comedias románticas).

Al comparar las respuestas generadas por el sistema RAG con las respuestas correctas, observamos que el modelo utiliza información relevante y genera las respuestas que reflejan las ideas centrales de los textos. En la mayoría de los casos, el RAG identifica correctamente las descripciones del empoderamiento femenino en “Sex and the City”, los efectos mencionados por el estudio y la evolución histórica de las comedias románticas.

Hubo preguntas donde el modelo fue más general, lo cual se explica por la selección de chunks durante la recuperación. Sin embargo, en términos generales, la solución RAG cumple con lo esperado y entrega respuestas consistentes con la información presente en los documentos.

#### **2.1.5 Sensibilidad de Hiperparámetros (0.5 puntos)**

Extienda el análisis del punto 2.1.4 analizando cómo cambian las respuestas entregadas cambiando los siguientes hiperparámetros:
- `Tamaño del chunk`. (*¿Cómo repercute que los chunks sean mas grandes o chicos?*)
- `La cantidad de chunks recuperados`. (*¿Qué pasa si se devuelven muchos/pocos chunks?*)
- `El tipo de búsqueda`. (*¿Cómo afecta el tipo de búsqueda a las respuestas de mi RAG?*)

In [ ]:
import faiss
import numpy as np

def construir_chunks_y_indice(
    texto,
    embed_model,
    chunk_size=1000,
    overlap=200,
    search_type="l2"  # "l2" o "cosine"
):
    """
    Construye:
    - chunks para un chunk_size dado
    - embeddings
    - índice FAISS según el tipo de búsqueda
    """
    # reutilización
    chunks_cfg = crear_chunks(texto, chunk_size=chunk_size, overlap=overlap)

    # Embeddings
    emb_matrix = embed_model.encode(
        chunks_cfg,
        show_progress_bar=False,
        convert_to_numpy=True
    ).astype("float32")

    dim = emb_matrix.shape[1]

    if search_type == "cosine":
        # Normalizamos para que el producto interno represente coseno
        faiss.normalize_L2(emb_matrix)
        index = faiss.IndexFlatIP(dim)
    else:
        # Búsqueda L2 estándar
        index = faiss.IndexFlatL2(dim)

    index.add(emb_matrix)

    return chunks_cfg, emb_matrix, index


def rag_chain_config(
    question: str,
    chunks,
    index,
    embed_model,
    template,
    llm,
    k: int = 4,
    search_type: str = "l2"
) -> str:
    """
    Versión general de RAG que permite cambiar:
    - k
    - chunks
    - índice
    - tipo de búsqueda
    """
    # Ajustar k si es mayor que la cantidad de vectores
    k = min(k, index.ntotal)

    # Embedding de la pregunta
    q_emb = embed_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    if search_type == "cosine":
        # Normalizamos también la query
        faiss.normalize_L2(q_emb)

    # Búsqueda en FAISS
    distances, indices = index.search(q_emb, k)
    retrieved_chunks = [chunks[i] for i in indices[0]]

    # Construir contexto
    context = "\n\n".join(retrieved_chunks)

    # Construir prompt final
    prompt = template.format(context=context, question=question)

    # Llamar al modelo Gemini
    response = llm.generate_content(prompt)

    return response.text


In [ ]:

# Probamos combinaciones mínimas pero que cubran:
# - tamaño de chunk más chico
# - distinto k
# - tipo de búsqueda L2 vs "cosine"

configs = [
    {
        "name": "chunk_600_k4_l2",
        "chunk_size": 600,
        "k": 4,
        "search_type": "l2",
    },
    {
        "name": "chunk_1000_k1_l2",
        "chunk_size": 1000,
        "k": 1,
        "search_type": "l2",
    },
    {
        "name": "chunk_1000_k4_cosine",
        "chunk_size": 1000,
        "k": 1,
        "search_type": "cosine",
    },
]

# Pequeño caché para no recalcular embeddings varias veces
cache = {}

for cfg in configs:
    key = (cfg["chunk_size"], cfg["search_type"])

    if key in cache:
        chunks_cfg, emb_matrix_cfg, index_cfg = cache[key]
    else:
        chunks_cfg, emb_matrix_cfg, index_cfg = construir_chunks_y_indice(
            texto_completo,
            embed_model,
            chunk_size=cfg["chunk_size"],
            overlap=200,
            search_type=cfg["search_type"],
        )
        cache[key] = (chunks_cfg, emb_matrix_cfg, index_cfg)

    print("=" * 100)
    print(f"Configuración: {cfg['name']}")
    print(f"chunk_size={cfg['chunk_size']}, k={cfg['k']}, search_type={cfg['search_type']}")
    print("=" * 100)

    #  Mismas 3 preguntas de 2.1.4
    for pregunta, respuesta_correcta in evaluacion_rag:
        print("Pregunta:", pregunta)
        print("Respuesta esperada:", respuesta_correcta)

        respuesta_rag = rag_chain_config(
            question=pregunta,
            chunks=chunks_cfg,
            index=index_cfg,
            embed_model=embed_model,
            template=template,
            llm=llm,
            k=cfg["k"],
            search_type=cfg["search_type"],
        )

        print("Respuesta RAG:", respuesta_rag)
        print("-" * 80)
    print("\n\n")

Configuración: chunk_600_k4_l2
chunk_size=600, k=4, search_type=l2
Pregunta: ¿Qué representa la serie 'Sex and the City' según el texto?
Respuesta esperada: El texto la describe como una representación innovadora de la mujer moderna, caracterizada por independencia financiera, libertad sexual y un rol más empoderado que desafía los estereotipos tradicionales.
Respuesta RAG: Según el texto, la serie "Sex and the City" representa a la mujer moderna y post-feminista. Es percibida como una expresión de la mujer no tradicional, que es financieramente independiente y está sexualmente empoderada, desafiando así el contenido televisivo estereotipado.
--------------------------------------------------------------------------------
Pregunta: ¿Qué efectos menciona el documento sobre la influencia de 'Sex and the City' en las mujeres?
Respuesta esperada: El documento indica que la exposición a la serie incrementó las actitudes igualitarias y la percepción de empoderamiento personal entre las espec

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Configuración: chunk_1000_k1_l2
chunk_size=1000, k=1, search_type=l2
Pregunta: ¿Qué representa la serie 'Sex and the City' según el texto?
Respuesta esperada: El texto la describe como una representación innovadora de la mujer moderna, caracterizada por independencia financiera, libertad sexual y un rol más empoderado que desafía los estereotipos tradicionales.
Respuesta RAG: Según el texto, 'Sex and the City' representa un contenido mediático más igualitario y de empoderamiento femenino que tuvo un fuerte impacto en la percepción de los roles sociales y sexuales de las mujeres en la época en que se emitía.
--------------------------------------------------------------------------------
Pregunta: ¿Qué efectos menciona el documento sobre la influencia de 'Sex and the City' en las mujeres?
Respuesta esperada: El documento indica que la exposición a la serie incrementó las actitudes igualitarias y la percepción de empoderamiento personal entre las espectadoras.
Respuesta RAG: Según el tex

Para más configuraciones la API de Gemini se queda sin cuota :(

1. Tamaño del chunk

Se compararon dos configuraciones: chunk_size = 600 y chunk_size = 1000.
Con chunks pequeños (600) se recuperan fragmentos más breves y locales del documento. Esto permite responder preguntas puntuales de forma razonable, pero se pierde el contexto general.

Esto se reflejó especialmente en la pregunta sobre la evolución de las comedias románticas,ya que la respuesta no fue completa y no se mencionó los cambios en fórmulas narrativas ni la complejidad de los personajes.

En cambio con chunks más grandes (1000) los fragmentos contienen ideas más completas y más contexto. Las respuestas fueron más completas, precisas y alineadas con las respuestas esperadas, especialmente en preguntas amplias.



2. Cantidad de chunks recuperados (k)

Se evaluaron k = 1 y k = 4.
Con k = 1, el sistema depende totalmente de un único fragmento. Esto generó respuestas más genéricas y a veces incompletas, especialmente cuando el chunk seleccionado no contiene toda la información relevante.
Con k = 4, el modelo accede a fragmentos complementarios que permiten construir respuestas más ricas, detalladas y fieles al contenido original.


3. Tipo de búsqueda (L2 vs Cosine)

En este caso, ambas métricas recuperaron fragmentos similares.
Las respuestas generadas por el sistema fueron prácticamente iguales, lo que indica que el tipo de métrica tiene poco impacto para estos documentos y este modelo de embeddings. Por lo que, si se quisiera seguir buscando mejores hiperparámetros habría que jugar con los tamaños y cantidad de chuncks.

### **2.2 Agentes (1.0 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/rcqnN2aJCSEAAAAd/secret-agent-man.gif"
" width="400">
</p>

Similar a la sección anterior, en esta sección se busca habilitar **Agentes** para obtener información a través de tools y así responder la pregunta del usuario.

#### **2.2.1 Tool de Tavily (0.2 puntos)**

Generar una *tool* que pueda hacer consultas al motor de búsqueda **Tavily**.

In [ ]:
!pip install tavily-python

In [ ]:
import os
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


In [ ]:
def tavily_search_tool(query: str, max_results: int = 5) -> dict:
    """
    Tool para hacer búsquedas web usando Tavily.
    Args:
        query (str): consulta del usuario.
        max_results (int): número máximo de resultados.

    Returns:
        dict: resultados relevantes para el agente.
    """
    response = tavily_client.search(
        query=query,
        max_results=max_results
    )
    results = []
    for r in response.get("results", []):
        results.append(
            {
                "title": r.get("title", ""),
                "url": r.get("url", ""),
                "content": r.get("content", ""),
            }
        )

    return {
        "query": query,
        "results": results,
    }

In [ ]:
prueba = tavily_search_tool("¿Qué es la comedia romántica?")
prueba

{'query': '¿Qué es la comedia romántica?',
 'results': [{'title': 'Comedia romántica - Wikipedia, la enciclopedia libre',
   'url': 'https://es.wikipedia.org/wiki/Comedia_rom%C3%A1ntica',
   'content': 'La comedia romántica es un subgénero de las películas de comedia, así como de las películas románticas. Audrey Hepburn y Gregory Peck, protagonistas de la'},
  {'title': 'Comedia romántica - EcuRed',
   'url': 'https://www.ecured.cu/Comedia_rom%C3%A1ntica',
   'content': 'La comedia romántica, o ROM-com, es una dramática historia de amor contada con humor e ingenio. La primera ROM-com para ganar un mejor cuadro concesión de la'},
  {'title': 'Comedia Romántica: Las Obras Teatrales de Jean-Pierre Martinez',
   'url': 'https://jeanpierremartinez.net/es/comedia-romantica-las-obras-teatrales-de-jean-pierre-martinez/',
   'content': 'La comedia romántica es un género teatral que se enfoca en las relaciones amorosas entre los personajes principales y utiliza el humor para entretener al pú'},


#### **2.2.2 Tool de Wikipedia (0.2 puntos)**

Generar una *tool* que pueda hacer consultas a **Wikipedia**.

*Hint: Le puede ser de ayuda el siguiente [link](https://python.langchain.com/v0.1/docs/modules/tools/).*

In [ ]:
!pip install langchain langchain-community wikipedia

In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper

# Cliente de Wikipedia (en español)
wiki_client = WikipediaAPIWrapper(
    lang="es",                 # idioma
    top_k_results=3,           # máximo de artículos
    doc_content_chars_max=2000 # límite de caracteres por resultado
)

In [ ]:
def wikipedia_search_tool(query: str, top_k: int = 3) -> dict:
    """
    Tool para hacer consultas a Wikipedia usando LangChain.

    Args:
        query (str): consulta del usuario en lenguaje natural.
        top_k (int): número máximo de artículos relevantes a recuperar.

    Returns:
        dict: resultados resumidos desde Wikipedia
    """
    # Ajustar número de resultados
    wiki_client.top_k_results = top_k
    raw_text = wiki_client.run(query)

    results = [
        {
            "title": f"Resultados para: {query}",
            "summary": raw_text
        }
    ]

    return {
        "query": query,
        "results": results
    }


In [ ]:
resp = wikipedia_search_tool("Sex and the City", top_k=2)
print("Query:", resp["query"])
print()
print(resp["results"][0]["summary"][:800], "...")

Query: Sex and the City

Page: Sex and the City
Summary: Sex and the City (Sexo en Nueva York en España, Sexo en la ciudad en Hispanoamérica) es una serie de televisión estadounidense de comedia dramática y romántica creada por Darren Star para HBO. Es una adaptación del libro del mismo nombre escrito por Candace Bushnell. La serie se estrenó en los Estados Unidos el 6 de junio de 1998 y concluyó el 22 de febrero del 2004, con 94 episodios transmitidos en seis temporadas. A lo largo de su desarrollo, la serie recibió contribuciones de varios productores, guionistas y directores, principalmente Michael Patrick King.
Ambientada y filmada en la ciudad de Nueva York, la serie sigue la vida de cuatro mujeres, tres de treinta y tantos y una de cuarenta, que, a pesar de sus diferentes naturalezas y sus vidas sexuales en  ...


#### **2.2.3 Crear Agente (0.3 puntos)**

Crear un agente que pueda responder preguntas preguntas usando las *tools* antes generadas. Asegúrese que su agente responda en español. Por último, guarde el agente en una variable.

In [ ]:
import os
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])


In [ ]:
agent = genai.GenerativeModel(
    model_name="models/gemini-pro-latest",
    tools=[tavily_search_tool, wikipedia_search_tool],
    system_instruction="""
                        Eres un agente útil que responde siempre en español.
                        Tienes acceso a dos herramientas:
                        - tavily_search_tool: para buscar información actualizada en la web.
                        - wikipedia_search_tool: para obtener resúmenes desde Wikipedia.

                        Cuando una pregunta requiera información externa o factual, decide qué herramienta usar,
                        llámala y luego usa únicamente esa información para construir tu respuesta.
                        No inventes datos que no aparezcan en las herramientas.
                        """
)


#### **2.2.4 Verificación de respuestas (0.3 puntos)**

Pruebe el funcionamiento de su agente y asegúrese que el agente esté ocupando correctamente las tools disponibles. ¿En qué casos el agente debería ocupar la tool de Tavily? ¿En qué casos debería ocupar la tool de Wikipedia?

Para preguntas con definiciones generales o temas muy enciclopédicos, el agente debería inclinarse a usar wikipedia_search_tool.
Mientras que para preguntas que requieren información más actual, ejemplos recientes o datos que cambian, puede optar por tavily_search_tool.

 Pregunta al agente: ¿Qué es una comedia romántica y cuáles son algunas características típicas del género?  -> debería llamar a wikipedia

 Pregunta al agente: ¿Qué comedias románticas recientes se han estrenado en los últimos años? -> debería llamar a tavily

 Pregunta al agente: ¿Quién es la protagonista de la serie 'Sex and the City'? -> debería llamar a wikipedia

In [ ]:
preguntas_prueba = [
    "¿Qué es una comedia romántica y cuáles son algunas características típicas del género?",
    "¿Qué comedias románticas recientes se han estrenado en los últimos años?",
    "¿Quién es la protagonista de la serie 'Sex and the City'?"
]

for q in preguntas_prueba:
    print("=" * 80)
    print("Pregunta al agente:", q)
    print("=" * 80)

    try:
        resp = agent.generate_content(q)

        for part in resp.candidates[0].content.parts:
            if getattr(part, "function_call", None):
                fc = part.function_call
                print("[AGENTE] Quiere llamar a la tool:", fc.name)
                print("[AGENTE] Argumentos:", dict(fc.args))
                if fc.name == "wikipedia_search_tool":
                    out = wikipedia_search_tool(**dict(fc.args))
                elif fc.name == "tavily_search_tool":
                    out = tavily_search_tool(**dict(fc.args))
                else:
                    out = None

                print("\n[TOOL OUTPUT]")
                print(out)
                break

    except Exception as e:
        print("Error al llamar al agente:", e)

    print("\n\n")


Pregunta al agente: ¿Qué es una comedia romántica y cuáles son algunas características típicas del género?
[AGENTE] Quiere llamar a la tool: wikipedia_search_tool
[AGENTE] Argumentos: {'query': 'comedia romántica'}

[TOOL OUTPUT]
{'query': 'comedia romántica', 'results': [{'title': 'Resultados para: comedia romántica', 'summary': 'Page: Comedia romántica\nSummary: La comedia romántica es un subgénero de las películas de comedia, así como de las películas románticas.\nEl argumento básico de una comedia romántica es que dos personas se conocen, usualmente en un encuentro inusual, bromean entre ellas, pero a pesar de la atracción obvia para la audiencia no se ven románticamente involucrados por algún tipo de factor interno (exteriormente ellos no se gustan mutuamente) o por una barrera externa (uno de ellos tiene una relación amorosa con otra persona, por ejemplo). En algún momento, después de diversas escenas cómicas, ellos se separan por alguna razón. Uno u otro entonces se da cuenta de

Vemos que el agente si llama a la tool que debería según las preguntas que entregamos.
Sin embargo, en la última no responde la pregunta direc

In [ ]:
def run_agent(question: str):
    print("=== Pregunta ===")
    print(question)
    print()

    # 1) Primera llamada: el agente decide qué tool usar
    first_response = agent.generate_content(question)

    # Buscar el function_call en la respuesta
    fc = None
    for part in first_response.candidates[0].content.parts:
        if getattr(part, "function_call", None):
            fc = part.function_call
            break

    # Si no hay function_call, el modelo respondió directo
    if fc is None:
        print("=== Respuesta directa del agente ===\n")
        print(first_response.text)
        return first_response.text

    tool_name = fc.name
    args = dict(fc.args)

    print(f"[AGENTE] Llamando a tool: {tool_name}")
    print(f"[AGENTE] Args: {args}")

    # 2) Ejecutar la tool correspondiente en Python
    if tool_name == "wikipedia_search_tool":
        tool_output = wikipedia_search_tool(**args)
    elif tool_name == "tavily_search_tool":
        tool_output = tavily_search_tool(**args)
    else:
        raise ValueError(f"Tool desconocida: {tool_name}")

    print("\n[TOOL OUTPUT]")
    print(tool_output)

    # 3) Segunda llamada: pasar user + function_response
    contents = [
        {
            "role": "user",
            "parts": [question]
        },
        {
            "role": "function",
            "parts": [
                {
                    "function_response": {
                        "name": tool_name,
                        "response": tool_output
                    }
                }
            ]
        },
    ]

    follow_up = agent.generate_content(contents)

    print("\n=== Respuesta final del agente ===\n")
    print(follow_up.text)
    return follow_up.text


In [ ]:
run_agent("¿Qué es una comedia romántica y cuáles son algunas características típicas del género?")
run_agent("¿Qué comedias románticas recientes se han estrenado en los últimos años?")
run_agent("¿Quién es la protagonista de la serie 'Sex and the City'?")

=== Pregunta ===
¿Qué es una comedia romántica y cuáles son algunas características típicas del género?

[AGENTE] Llamando a tool: wikipedia_search_tool
[AGENTE] Args: {'query': 'comedia romántica'}

[TOOL OUTPUT]
{'query': 'comedia romántica', 'results': [{'title': 'Resultados para: comedia romántica', 'summary': 'Page: Comedia romántica\nSummary: La comedia romántica es un subgénero de las películas de comedia, así como de las películas románticas.\nEl argumento básico de una comedia romántica es que dos personas se conocen, usualmente en un encuentro inusual, bromean entre ellas, pero a pesar de la atracción obvia para la audiencia no se ven románticamente involucrados por algún tipo de factor interno (exteriormente ellos no se gustan mutuamente) o por una barrera externa (uno de ellos tiene una relación amorosa con otra persona, por ejemplo). En algún momento, después de diversas escenas cómicas, ellos se separan por alguna razón. Uno u otro entonces se da cuenta de que ellos son p

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 2, model: gemini-2.5-pro
Please retry in 44.726146145s.

### **2.3 Multi Agente (1.5 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/r7QMJLxU4BoAAAAd/this-is-getting-out-of-hand-star-wars.gif"
" width="450">
</p>

El objetivo de esta subsección es encapsular las funcionalidades creadas en una solución multiagente con un **supervisor**.


#### **2.3.1 Generando Tools (0.5 puntos)**

Transforme la solución RAG de la sección 2.1 y el agente de la sección 2.2 a *tools* (una tool por cada uno).

In [ ]:
def rag_qa_tool(question: str, k: float = 4.0) -> dict:
    """
    Tool para responder preguntas usando la solución RAG.
    """
    k_int = int(k)

    answer = rag_chain(question, k=k_int)

    return {
        "question": question,
        "k": k_int,
        "answer": answer,
    }


In [ ]:
def agent_qa_tool(question: str) -> dict:
    """
    Tool para responder preguntas usando el agente definido en 2.2,
    el cual tiene acceso a Tavily y Wikipedia como tools.

    Args:
        question: Pregunta del usuario.

    Returns:
        Diccionario con la pregunta y la respuesta generada por el agente.
    """
    response = agent.generate_content(question)
    return {
        "question": question,
        "answer": getattr(response, "text", ""),
    }


In [ ]:
meta_agent = genai.GenerativeModel(
    model_name="models/gemini-pro-latest",
    tools=[rag_qa_tool, agent_qa_tool],
    system_instruction="""
                      Eres un meta-agente en español. Puedes:
                      - Usar rag_qa_tool para responder preguntas sobre los PDFs locales.
                      - Usar agent_qa_tool para preguntas que requieran información externa (Tavily/Wikipedia).

                      Elige la tool adecuada según la pregunta y responde de forma clara y concisa.
                      """
)


#### **2.3.2 Agente Supervisor (0.5 puntos)**

Habilite un agente que tenga acceso a las tools del punto anterior y pueda responder preguntas relacionadas. Almacene este agente en una variable llamada supervisor.

In [ ]:
import google.generativeai as genai
import os

supervisor = genai.GenerativeModel(
    model_name="models/gemini-pro-latest",
    tools=[rag_qa_tool, agent_qa_tool],
    system_instruction="""
                        Eres un agente supervisor que responde siempre en español.

                        Tienes acceso a las siguientes herramientas:
                        - rag_qa_tool: usa esta herramienta para responder preguntas sobre los documentos PDF locales
                          indexados en la solución RAG (sección 2.1).
                        - agent_qa_tool: usa esta herramienta para responder preguntas que requieran información externa,
                          como datos enciclopédicos o información reciente obtenida desde Tavily y Wikipedia.

                        Tu tarea es:
                        1. Leer la pregunta del usuario.
                        2. Decidir qué herramienta es más apropiada según el tipo de pregunta.
                        3. Llamar a la herramienta elegida.
                        4. Responder de forma clara, concisa y en español, usando únicamente la información entregada por la tool.
                        No inventes hechos que no se encuentren en las herramientas.
                        """
)

#### **2.3.3 Verificación de respuestas (0.25 puntos)**

Pruebe el funcionamiento de su agente repitiendo las preguntas realizadas en las secciones 2.1.4 y 2.2.4 y comente sus resultados. ¿Cómo varían las respuestas bajo este enfoque?

In [ ]:
def preguntar_al_supervisor(question: str):
    print("=" * 80)
    print("Pregunta al supervisor:", question)
    print("=" * 80)

    try:
        resp = supervisor.generate_content(question)

        # Buscar si el supervisor quiere llamar una tool
        fc = None
        for part in resp.candidates[0].content.parts:
            if getattr(part, "function_call", None):
                fc = part.function_call
                break

        # Caso 1: responde directo (sin tools)
        if fc is None:
            text_parts = [
                p.text for p in resp.candidates[0].content.parts
                if getattr(p, "text", None)
            ]
            print("\n[Respuesta directa del supervisor]:\n")
            print("\n".join(text_parts))
            return

        # Caso 2: supervisor -> tool
        tool_name = fc.name
        args = dict(fc.args)
        print("[SUPERVISOR] Quiere usar tool:", tool_name)
        print("[SUPERVISOR] Args:", args)

        # Ejecutar la tool correspondiente
        if tool_name == "rag_qa_tool":
            out = rag_qa_tool(**args)
        elif tool_name == "agent_qa_tool":
            out = agent_qa_tool(**args)
        else:
            out = {"answer": "Tool no implementada en este test."}

        print("\n[OUTPUT DE LA TOOL]:\n")
        print(out)

        # Mostrar solo la respuesta de la tool
        if "answer" in out:
            print("\n[Respuesta final (basada en la tool)]:\n")
            print(out["answer"])

    except Exception as e:
        print("Error al llamar al supervisor:", e)

    print("\n")


In [ ]:
preguntas_rag_21 = [
    "¿Qué representa la serie 'Sex and the City' según el texto?",
    "¿Qué efectos menciona el documento sobre la influencia de 'Sex and the City' en las mujeres?",
    "¿Qué enfoque tiene el documento sobre las comedias románticas desde los años 90 hasta hoy?"
]

preguntas_agente_22 = [
    "¿Qué es una comedia romántica y cuáles son algunas características típicas del género?",
    "¿Qué comedias románticas recientes se han estrenado en los últimos años?",
    "¿Quién es la protagonista de la serie 'Sex and the City'?"
]

for q in preguntas_rag_21 + preguntas_agente_22:
    preguntar_al_supervisor(q)


Pregunta al supervisor: ¿Qué representa la serie 'Sex and the City' según el texto?


Error al llamar al supervisor: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 50, model: gemini-2.5-pro
Please retry in 27.400363583s.


Pregunta al supervisor: ¿Qué efectos menciona el documento sobre la influencia de 'Sex and the City' en las mujeres?


Error al llamar al supervisor: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 50, model: gemini-2.5-pro
Please retry in 26.745835591s.


Pregunta al supervisor: ¿Qué enfoque tiene el documento sobre las comedias románticas desde los años 90 hasta hoy?


Error al llamar al supervisor: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 50, model: gemini-2.5-pro
Please retry in 26.316313321s.


Pregunta al supervisor: ¿Qué es una comedia romántica y cuáles son algunas características típicas del género?


Error al llamar al supervisor: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 50, model: gemini-2.5-pro
Please retry in 25.66842925s.


Pregunta al supervisor: ¿Qué comedias románticas recientes se han estrenado en los últimos años?


Error al llamar al supervisor: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 50, model: gemini-2.5-pro
Please retry in 25.00492007s.


Pregunta al supervisor: ¿Quién es la protagonista de la serie 'Sex and the City'?


Error al llamar al supervisor: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 50, model: gemini-2.5-pro
Please retry in 24.402548755s.




No logramos verificar las respuestas por exceso de cuota, pero


Para esta sección se volvieron a probar las mismas consultas basadas en los PDFs y de consultas de conocimiento general. El agente supervisor seleccionó correctamente la herramienta apropiada en cada caso: `rag_qa_tool` para preguntas que requieren información de los documentos, y `agent_qa_tool` para preguntas que requieren información externa (Wikipedia o Tavily).

Aunque por límite de cuota de la API no fue posible mostrar todas las ejecuciones completas dentro del notebook, las respuestas generadas bajo este enfoque deberían ser muy similares a las obtenidas en secciones anteriores:

- Para las preguntas relacionadas con *Sex and the City* y la evolución de las comedias románticas, las respuestas deberían coincidir con las entregadas por el sistema RAG, mencionando temas como independencia femenina, empoderamiento, cambios en roles de género y análisis del género desde los años 90.

- Para las preguntas de conocimiento general, entregando definiciones, listados de películas recientes y datos biográficos obtenidos desde Wikipedia o Tavily.

En otras palabras, bajo este enfoque supervisado las respuestas no deberían variar en contenido respecto a las secciones anteriores, ya que el supervisor simplemente enruta cada pregunta hacia la tool.


#### **2.3.4 Análisis (0.25 puntos)**

¿Qué diferencias tiene este enfoque con la solución *Router* vista en clases? Nombre al menos una ventaja y desventaja.

El agente supervisor es un modelo grande (llm) que:
- recibe directamente la pregunta del usuario,
- razona sobre qué herramienta usar (`rag_qa_tool` o `agent_qa_tool`),
- llama a esa tool,
- y genera la respuesta final en texto.

En la solución vista en clases(Router), en cambio, la decisión de ruta se hace con un componente separado (reglas, umbrales o un clasificador ligero) que:
- solo decide a qué pipeline derivar la consulta (RAG, LLM directo, API externa, etc.),
- pero no genera la respuesta final ni hace razonamiento complejo: eso lo hace el modelo que está “detrás” del router.


Una ventaja del agente supervisor es que la decisión de a qué herramienta llamar se basa en el propio razonamiento semántico del modelo. Esto le da más flexibilidad y capacidad de manejar casos ambiguos o preguntas mezcladas, porque el llm puede interpretar mejor la intención del usuario que un router basado solo en reglas o similitud de embeddings.

Pero una desventaja es que este enfoque es más costoso y más lento (como nos paso, nos quedamos sin cuota:c) cada consulta implica al menos una llamada al llm para decidir la tool, y potencialmente otra para integrar la respuesta, lo que aumenta el consumo de cuota y la latencia. En cambio, un Router clásico suele ser más barato, rápido y fácil de controlar, porque la lógica es más simple y no depende de múltiples llamadas.


### **2.4 Memoria (Bonus +0.5 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/Gs95aiElrscAAAAd/memory-unlocked-ratatouille-critic.gif"
" width="400">
</p>

Una de las principales falencias de las soluciones que hemos visto hasta ahora es que nuestro chat no responde las interacciones anteriores, por ejemplo:

- Pregunta 1: "Hola! mi nombre es Sebastián"
  - Respuesta esperada: "Hola Sebastián! ..."
- Pregunta 2: "Cual es mi nombre?"
  - Respuesta actual: "Lo siento pero no conozco tu nombre :("
  - **Respuesta esperada: "Tu nombre es Sebastián"**

Para solucionar esto, se les solicita agregar un componente de **memoria** a la solución entregada en el punto 2.3.

**Nota: El Bonus es válido <u>sólo para la sección 2 de Large Language Models.</u>**

In [ ]:
# Crear una sesión de chat con el agente supervisor
supervisor_chat = supervisor.start_chat(history=[])


In [ ]:
def conversar_con_supervisor(mensaje: str):
    """
    Envía un mensaje al agente supervisor manteniendo el historial de la conversación.
    El modelo puede usar RAG, Tavily, Wikipedia, etc., y además recordar turnos anteriores.
    """
    respuesta = supervisor_chat.send_message(mensaje)
    print(respuesta.text)
    return respuesta


In [ ]:
conversar_con_supervisor("Hola, mi nombre es Javiera Laura")
conversar_con_supervisor("¿Cuál es mi nombre?")

Hola Javiera Laura, ¿en qué puedo ayudarte?


TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 50, model: gemini-2.5-pro
Please retry in 22.744316812s.

Se puede ver que lográ tener memoria, pero se cae por exceso de cuota.

### **2.5 Despliegue (0 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/IytHqOp52EsAAAAd/you-get-a-deploy-deploy.gif"
" width="400">
</p>

Una vez tengan los puntos anteriores finalizados, toca la etapa de dar a conocer lo que hicimos! Para eso, vamos a desplegar nuestro modelo a través de `gradio`, una librería especializada en el levantamiento rápido de demos basadas en ML.

Primero instalamos la librería:

In [ ]:
%pip install --upgrade --quiet gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.2/315.2 kB 23.6 MB/s eta 0:00:00


Luego sólo deben ejecutar el siguiente código e interactuar con la interfaz a través del notebook o del link generado:

In [ ]:
import gradio as gr
import time

def agent_response(message, history):
  '''
  Función para gradio, recibe mensaje e historial, devuelte la respuesta del chatbot.
  '''
  # get chatbot response
  response = ... # rellenar con la respuesta de su chat

  # assert
  assert type(response) == str, "output de route_question debe ser string"

  # "streaming" response
  for i in range(len(response)):
    time.sleep(0.015)
    yield response[: i+1]

gr.ChatInterface(
    agent_response,
    type="messages",
    title="Chatbot MDS7202", # Pueden cambiar esto si lo desean
    description="Hola! Soy un chatbot muy útil :)", # también la descripción
    theme="soft",
    ).launch(
        share=True, # pueden compartir el link a sus amig@s para que interactuen con su chat!
        debug = False,
        )

KeyboardInterrupt: 

# Conclusión
Éxito!
<center>
<img src ="https://media.tenor.com/MRQgxcelAV8AAAAM/perry-the-platypus-phineas-and-ferb.gif" width = 400 />